In [1]:
!pip install -q pandas nltk networkx scikit-learn

In [11]:
import os
import re
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

def clean_sentence(s: str) -> str:
    """
    1) lowercase
    2) remove non-letters (keep spaces)
    3) squeeze extra spaces
    """
    s = s.lower()
    s = re.sub(r'[^a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
# read the excel; if your sheet name is custom, pass sheet_name="..."
df = pd.read_csv("tennis_articles.csv", encoding='latin1')

print("Columns:", df.columns.tolist())
df.head(3)

Columns: ['article_id', 'article_title', 'article_text', 'source']


,article_id,article_title,article_text,source
0,1,"I do not have friends in tennis, says Maria Sh...",Maria Sharapova has basically no friends as te...,https://www.tennisworldusa.org/tennis/news/Mar...
1,2,Federer defeats Medvedev to advance to 14th Sw...,"BASEL, Switzerland (AP)  Roger Federer advanc...",http://www.tennis.com/pro-game/2018/10/copil-s...
2,3,Tennis: Roger Federer ignored deadline set by ...,Roger Federer has revealed that organisers of ...,https://scroll.in/field/899938/tennis-roger-fe...


In [8]:
if "article_title" in df.columns:
    df = df.drop(columns=["article_title"])

In [9]:
# Try to detect a reasonable text column
possible_text_cols = ["article_text", "text", "content", "body"]
text_col = None
for c in df.columns:
    if c.lower() in possible_text_cols:
        text_col = c
        break

if text_col is None:
    # If we didn’t find one, just pick the first column that looks texty
    # (you can hardcode your column name here instead)
    text_col = df.columns[0]

print("Using text column:", text_col)

# Drop rows with missing/empty text
df = df[ df[text_col].astype(str).str.strip().ne("") ].reset_index(drop=True)
print("Rows after cleaning:", len(df))

Using text column: article_text
Rows after cleaning: 8


In [12]:
all_sentences = []
article_id_for_sentence = []  # keep track of where each sentence came from (optional)

for idx, row in df.iterrows():
    text = str(row[text_col])
    # NLTK sentence split
    sents = sent_tokenize(text)
    for s in sents:
        # ignore super-short sentences
        if len(s.strip().split()) >= 3:
            all_sentences.append(s.strip())
            article_id_for_sentence.append(idx)

print("Total sentences:", len(all_sentences))
print(all_sentences[:5])

Total sentences: 130
['Maria Sharapova has basically no friends as tennis players on the WTA Tour.', "The Russian player has no problems in openly speaking about it and in a recent interview she said: 'I don't really hide any feelings too much.", 'I think everyone knows this is my job here.', "When I'm on the courts or when I'm on the court playing, I'm a competitor and I want to beat every single person whether they're in the locker room or across the net.", "So I'm not the one to strike up a conversation about the weather and know that in the next few minutes I have to go and try to win a tennis match."]


In [13]:
# Download once (safe to re-run)
if not os.path.exists("/content/glove.6B.100d.txt"):
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -O /content/glove.6B.zip
    !unzip -q /content/glove.6B.zip -d /content/

# Load into a dict: word -> 100-d vector (np.array)
glove_path = "/content/glove.6B.100d.txt"
emb_dim = 100
word2vec = {}

with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        w = parts[0]
        vec = np.asarray(parts[1:], dtype="float32")
        if vec.shape[0] == emb_dim:
            word2vec[w] = vec

print("Loaded words:", len(word2vec))

Loaded words: 400000


In [14]:
cleaned_sentences = []
for s in all_sentences:
    c = clean_sentence(s)
    # remove stopwords
    tokens = [t for t in c.split() if t not in STOPWORDS]
    cleaned_sentences.append(" ".join(tokens))

print("Example (orig → cleaned):")
for i in range(3):
    print("•", all_sentences[i])
    print("→", cleaned_sentences[i])
    print()

Example (orig → cleaned):
• Maria Sharapova has basically no friends as tennis players on the WTA Tour.
→ maria sharapova basically friends tennis players wta tour

• The Russian player has no problems in openly speaking about it and in a recent interview she said: 'I don't really hide any feelings too much.
→ russian player problems openly speaking recent interview said really hide feelings much

• I think everyone knows this is my job here.
→ think everyone knows job



In [15]:
def sentence_to_vector(sentence: str, emb_table: dict, dim: int = 100) -> np.ndarray:
    if not sentence:
        return np.zeros(dim, dtype="float32")
    tokens = sentence.split()
    if not tokens:
        return np.zeros(dim, dtype="float32")

    vecs = []
    for t in tokens:
        if t in emb_table:
            vecs.append(emb_table[t])
    if not vecs:
        return np.zeros(dim, dtype="float32")
    return np.mean(np.vstack(vecs), axis=0).astype("float32")

sent_vectors = np.vstack([
    sentence_to_vector(s, word2vec, emb_dim) for s in cleaned_sentences
])

sent_vectors.shape

(130, 100)

In [16]:
sim_matrix = cosine_similarity(sent_vectors)
sim_matrix.shape

(130, 130)

In [17]:
# Avoid self-loops dominating (optional): set diagonal to 0
np.fill_diagonal(sim_matrix, 0.0)

# Build graph
graph = nx.from_numpy_array(sim_matrix)

# PageRank (returns dict: node -> score)
scores = nx.pagerank(graph, max_iter=200, tol=1e-6)
len(scores), list(scores.items())[:3]

(130,
 [(0, 0.007346221154221762),
  (1, 0.007831540195639895),
  (2, 0.007188734016427924)])

In [18]:
def summarize(sentences, scores, top_n=10, keep_original_order=True):
    # scores is a dict: index -> score
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    chosen_ids = [i for i, _ in ranked[:top_n]]

    if keep_original_order:
        # return sentences in the order they originally appeared
        chosen_ids = sorted(chosen_ids)

    return [sentences[i] for i in chosen_ids]

summary = summarize(all_sentences, scores, top_n=10, keep_original_order=True)

print("=== SUMMARY (Top 10 sentences) ===\n")
for i, s in enumerate(summary, 1):
    print(f"{i:02d}. {s}")

=== SUMMARY (Top 10 sentences) ===

01. So I'm not the one to strike up a conversation about the weather and know that in the next few minutes I have to go and try to win a tennis match.
02. Speaking at the Swiss Indoors tournament where he will play in Sundays final against Romanian qualifier Marius Copil, the world number three said that given the impossibly short time frame to make a decision, he opted out of any commitment.
03. Major players feel that a big event in late November combined with one in January before the Australian Open will mean too much tennis and too little rest.
04. Currently in ninth place, Nishikori with a win could move to within 125 points of the cut for the eight-man event in London next month.
05. He used his first break point to close out the first set before going up 3-0 in the second and wrapping up the win on his first match point.
06. I felt like the best weeks that I had to get to know players when I was playing were the Fed Cup weeks or the Olympic